# D-Fire：Colab / Kaggle 共用入口
独立 HIFE-YOLO 重建，尚无真实训练精度结论。先开启 GPU；Kaggle 开启 Internet。
数据单元格可下载官方 README 推荐的 Kaggle 副本，并从自带 YAML 读取类别顺序；保留其现有划分。
输出请设为持久目录：Colab 挂载后的 Drive，或 Kaggle `/kaggle/working`（保存版本下载输出）。
不在 Notebook 填入或提交账号密钥。初次先跑 baseline，再跑 hife；保持数据清单和 seed 一致。


In [ ]:
import os, subprocess, sys
from pathlib import Path
REPO = Path('/kaggle/working/D-Fire' if Path('/kaggle').exists() else '/content/D-Fire')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Cranzz/D-Fire.git', str(REPO)], check=True)
os.chdir(REPO)
# 首次运行记录 commit；正式对比请将 PINNED_COMMIT 固定到同一已验证提交。
PINNED_COMMIT = None
if not PINNED_COMMIT:
    # 首次搭建时同步修复；正式实验应固定 PINNED_COMMIT。
    subprocess.run(['git', 'pull', '--ff-only'], check=True)
if PINNED_COMMIT:
    subprocess.run(['git', 'checkout', PINNED_COMMIT], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[test]'], check=True)
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)


In [ ]:
import torch
assert torch.cuda.is_available(), '请先开启 GPU'
print(torch.__version__, torch.cuda.get_device_name(0))
subprocess.run([sys.executable, '-m', 'pytest', '-q'], check=True)
subprocess.run([sys.executable, '-m', 'dfire.cli', 'audit', '--model', 'hife'], check=True)


In [ ]:
# Colab：挂载持久输出目录（首次会要求你在浏览器完成授权）。
IS_KAGGLE = Path('/kaggle').exists()
if not IS_KAGGLE:
    from google.colab import drive
    drive.mount('/content/drive')
OUTPUT = Path('/kaggle/working/dfire-runs' if IS_KAGGLE else '/content/drive/MyDrive/dfire-runs')
OUTPUT.mkdir(parents=True, exist_ok=True)

# 还没有数据：下载官方 D-Fire README 推荐的 Kaggle 副本。
# 公共数据通常不需要令牌；如平台要求认证，请在平台界面完成，不把密钥写进 Notebook。
subprocess.run([sys.executable, '-m', 'pip', 'install', 'kagglehub>=0.3,<1'], check=True)
import kagglehub
DATASET_HANDLE = 'sayedgamal99/smoke-fire-detection-yolo'
DATA_ROOT = Path(kagglehub.dataset_download(DATASET_HANDLE))
print('Downloaded data:', DATA_ROOT)
# 沿用下载包划分，从其 YAML 读取类别顺序，生成当前 Colab 的绝对路径配置。
# 该副本划分尚未核实等同 HIFE 原文，不能直接对照作者分数。
from dfire.prepare import prepare
DATA = prepare(DATA_ROOT, OUTPUT / 'data.yaml')
MODEL = 'baseline'  # 随后改为 hife；也支持 msghost / retention / ctd
SEED = 20260716
MANIFEST = OUTPUT / 'data-manifest.json'
subprocess.run([sys.executable, '-m', 'dfire.manifest', '--data', str(DATA), '--output', str(MANIFEST)], check=True)


In [ ]:
RUN = f'{MODEL}-s{SEED}'
subprocess.run([sys.executable, '-m', 'dfire.cli', 'train', '--model', MODEL,
    '--data', str(DATA), '--name', RUN, '--seed', str(SEED), '--project', str(OUTPUT), '--device', '0'], check=True)


## 中断恢复与最终评测
恢复仅适用于尚未跑满原计划 epoch 的 last.pt。修改路径后单独执行需要的单元格。
先验证集选模型；冻结结构后只对最终候选运行 test。指标不能与作者未知划分直接等同。


In [ ]:
# 中断后恢复：取消注释；不要与首次训练同时执行。
# LAST = OUTPUT / RUN / 'weights/last.pt'
# subprocess.run([sys.executable, '-m', 'dfire.cli', 'train', '--model', MODEL,
#     '--data', str(DATA), '--name', RUN, '--project', str(OUTPUT), '--resume', str(LAST)], check=True)


In [ ]:
subprocess.run([sys.executable, '-m', 'dfire.cli', 'eval', '--weights', str(OUTPUT / RUN / 'weights/best.pt'),
    '--data', str(DATA), '--split', 'val', '--output', str(OUTPUT / f'{RUN}-evaluation')], check=True)
